In [ ]:
from redshift_config import REDSHIFT_URL, S3_URL, AWS_ACCESS_KEY, AWS_SECRET_KEY, AWS_IAM_ROLE
from redshift_util import redshift_setting


In [ ]:
# 엑셀파일 다운로드 - 전처리 및 sv 파일로 변환 - S3에 로드
redshift_setting()

In [ ]:
%load_ext sql

In [ ]:
%sql {REDSHIFT_URL}

In [ ]:
%%sql
--schema 없을때만 실행
CREATE SCHEMA raw_data;
CREATE SCHEMA analytics;
CREATE SCHEMA adhoc;
CREATE SCHEMA pii;

In [ ]:
%%sql

DROP TABLE IF EXISTS raw_data.animal_hospital;
CREATE TABLE raw_data.animal_hospital (
    name varchar(256),
    phone_number varchar(256),
    addr varchar(256),
    id varchar(256) primary key,
    location_code varchar(256)
);


In [ ]:
animal_hospital_s3_url = f'{S3_URL}/animal_hospital.csv'
print(animal_hospital_s3_url)

In [ ]:
%%sql

COPY raw_data.animal_hospital
FROM '{animal_hospital_s3_url}'
credentials 'aws_iam_role={AWS_IAM_ROLE}'
delimiter ',' dateformat 'auto' timeformat 'auto' IGNOREHEADER 1 removequotes;


In [ ]:
%%sql

SELECT * from raw_data.animal_hospital limit 10;

In [ ]:
%%sql

DROP TABLE IF EXISTS raw_data.protected_animal;

CREATE TABLE raw_data.protected_animal (
    id varchar(256) primary key,
    animal_id varchar(256),
    animal_type varchar(256),
    breed varchar(256),
    fur_color varchar(256),
    sex varchar(256),
    neutering varchar(256),
    feature varchar(256),
    rescue_day timestamp,
    rescue_reason varchar(256),
    rescue_location varchar(256),
    notice_period varchar(256),
    protection_center varchar(256),
    exponent varchar(256),
    address varchar(256),
    phone_number varchar(256)
);

In [ ]:
protected_animal_s3_url = f'{S3_URL}/protected_animal.csv'
print(animal_hospital_s3_url)

In [ ]:
%%sql

COPY raw_data.protected_animal
FROM '{protected_animal_s3_url}'
credentials 'aws_iam_role={AWS_IAM_ROLE}'
delimiter ',' dateformat 'auto' timeformat 'auto' IGNOREHEADER 1 removequotes;

In [ ]:
%%sql

select * from raw_data.protected_animal limit 10;

In [ ]:
%%sql

DROP TABLE IF EXISTS analytics.protected_animal_location;
CREATE TABLE analytics.protected_animal_location AS
SELECT
    LEFT(id, 2) AS location,
    CASE location
        WHEN '서울' THEN 'KR-11'
        WHEN '부산' THEN 'KR-26'
        WHEN '대구' THEN 'KR-27'
        WHEN '인천' THEN 'KR-28'
        WHEN '광주' THEN 'KR-29'
        WHEN '대전' THEN 'KR-30'
        WHEN '울산' THEN 'KR-31'
        WHEN '세종' THEN 'KR-50'
        WHEN '경기' THEN 'KR-41'
        WHEN '강원' THEN 'KR-42'
        WHEN '충북' THEN 'KR-43'
        WHEN '충남' THEN 'KR-44'
        WHEN '전북' THEN 'KR-45'
        WHEN '전남' THEN 'KR-46'
        WHEN '경북' THEN 'KR-47'
        WHEN '경남' THEN 'KR-48'
        WHEN '제주' THEN 'KR-49'
    END AS region_code,
    COUNT(*) AS cnt_animal
FROM raw_data.protected_animal
GROUP BY location;

In [ ]:
%%sql

DROP TABLE IF EXISTS raw_data.to_adopt_animal;

CREATE TABLE raw_data.to_adopt_animal (
    id varchar(256) primary key,
    animal_id varchar(256),
    breed varchar(256),
    color varchar(256),
    sex varchar(256),
    neutering varchar(256),
    age_weight varchar(256),
    feature_rescue varchar(256),
    feature_social varchar(256),
    feature_health varchar(256),
    rescue_location varchar(256),
    reception_dt timestamp,
    etc varchar(256),
    center varchar(256),
    status varchar(256),
    protection_center varchar(256),
    phone_number varchar(256),
    protection_addr varchar(256)
);

In [ ]:
to_adopt_animal_s3_url = f'{S3_URL}/to_adopt_animal.csv'
print(to_adopt_animal_s3_url)

In [ ]:
%%sql

COPY raw_data.to_adopt_animal
FROM '{to_adopt_animal_s3_url}'
credentials 'aws_iam_role={AWS_IAM_ROLE}'
delimiter ',' dateformat 'auto' timeformat 'auto' IGNOREHEADER 1 removequotes;

In [ ]:
%%sql
SELECT * from raw_data.to_adopt_animal limit 10;

In [ ]:
%%sql

DROP TABLE IF EXISTS analytics.to_adopt_animal_location;
CREATE TABLE analytics.to_adopt_animal_location AS
SELECT
    LEFT(id, 2) AS location,
    CASE location
        WHEN '서울' THEN 'KR-11'
        WHEN '부산' THEN 'KR-26'
        WHEN '대구' THEN 'KR-27'
        WHEN '인천' THEN 'KR-28'
        WHEN '광주' THEN 'KR-29'
        WHEN '대전' THEN 'KR-30'
        WHEN '울산' THEN 'KR-31'
        WHEN '세종' THEN 'KR-50'
        WHEN '경기' THEN 'KR-41'
        WHEN '강원' THEN 'KR-42'
        WHEN '충북' THEN 'KR-43'
        WHEN '충남' THEN 'KR-44'
        WHEN '전북' THEN 'KR-45'
        WHEN '전남' THEN 'KR-46'
        WHEN '경북' THEN 'KR-47'
        WHEN '경남' THEN 'KR-48'
        WHEN '제주' THEN 'KR-49'
    END AS region_code,
    COUNT(*) AS cnt_animal
FROM raw_data.to_adopt_animal
GROUP BY location;

In [ ]:
%%sql
SELECT * from analytics.to_adopt_animal_location;